## Scrapeando datos

Web: [todostuslibros.com](https://www.todostuslibros.com/busquedas?editorial=planeta)

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import csv
import time

opciones = Options()
opciones.add_argument("--headless")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=opciones)

para el siguiente: link + #fichaTecnica

In [ ]:
# ======================================================================================
# VERSIÓN FINAL (todas las editoriales)
# ======================================================================================


# Creando driver
def create_driver():
    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=opciones)


# Proceso para pequeñas/medianas
def search_editorial(editorial_id, editorial_nombre, pag_inicio, pag_final):
    libros = []
    page_num = pag_inicio

    while page_num <= pag_final:
        url = f"https://www.todostuslibros.com/editoriales/{editorial_id}/catalogo?page={page_num}"
        
        if page_num < 10:
            print(f"Página {page_num}...")
        elif page_num == 10:
            print("Página 10 y más...")
        else:
            pass
        
        driver.get(url)
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select("h2 a")
        
        if not items:
            print("Sin más resultados.")
            break

        for h2 in soup.select("h2"):
            a = h2.find("a")
            if not a:
                continue
            titulo = a.get_text(strip=True)

            h3 = h2.find_next_sibling("h3")
            autor = h3.get_text(strip=True) if h3 else ""

            precio_tag = h2.find_next("strong")
            precio = precio_tag.get_text(strip=True) if precio_tag else ""



            url_libro = a["href"] if a.get("href") else ""

            libros.append({
                "editorial": editorial_nombre,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        page_num += 1


    return libros

# Proceso para grandes
def search_big_editorial(editorial_id, editorial_nombre, anio_inicio, anio_final):
    libros = []

    for anio in range(anio_inicio, anio_final + 1):
        print(f"Año {anio}...")
        page_num = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{editorial_id}/catalogo?anio={anio}&page={page_num}"
            driver.get(url)
            time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            items = soup.select("h2 a")

            if not items:
                print(f"  Sin más resultados en {anio}, página {page_num}.")
                break  # pasa al año siguiente

            for h2 in soup.select("h2"):
                a = h2.find("a")
                if not a:
                    continue
                titulo = a.get_text(strip=True)

                h3 = h2.find_next_sibling("h3")
                autor = h3.get_text(strip=True) if h3 else ""

                precio_tag = h2.find_next("strong")
                precio = precio_tag.get_text(strip=True) if precio_tag else ""

                url_libro = a["href"] if a.get("href") else ""

                libros.append({
                    "editorial": editorial_nombre,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            page_num += 1
            time.sleep(2)

    return libros


# Leyendo links de libros (general)
def extract_data(url):

    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Accediendo a las fichas de datos técnicos
    items = soup.find_all("dl", class_ = "datos-tecnicos")

    # Listas donde se guardarán los datos de cada iteración
    nombres = []
    datos = []

    # Extrayendo datos
    for item in items:
        # Listas con los containers
        # dt: nombre del dato técnico; dd: contenido del dato técnico
        nombre_dato, contenido_dato = item.find_all('dt'), item.find_all('dd')

        # Extrayendo los nombres de los datos técnicos
        for nombre in nombre_dato:
            nombres.append(nombre.get_text(strip = True).replace(":", "").strip())

        # Extrayendo contenidos de los datos técnicos
        # El contenido está en una etiqueta a, span o directamente en el dd (según se ha visto en el código)
        for dato in contenido_dato:

            # Contenido en etiqueta a
            a = dato.find_all("a")

            if not a:
                pass

            else:
                # Hay contenidos que tienen más de un dato (categorías por ejemplo)
                lista = []
                for clase in a:
                    texto = clase.get_text(strip = True)
                    lista.append(texto)
                
                if len(lista) == 1:
                    datos.append(lista[0])
                else:
                    datos.append(lista)

                continue
            
            # Si la etiqueta no es a, se prueba con span
            span = dato.find("span")

            if not span:
                pass

            else:
                texto = span.get_text(strip = True)
                datos.append(texto)

                continue
            
            # Si la etiqueta tampoco está en span, se extrae el texto directamente
            texto = dato.get_text()
            datos.append(" ".join(texto.split()))

    # Añadiendo sinopsis
    sinopsis = soup.find("div", id="collapseSynopsis")
    nombres.append("Sinopsis")
    if sinopsis:
        parrafos = [p.get_text(strip=True) for p in sinopsis.find_all("p")]
        datos.append(" | ".join(parrafos))
    else:
        datos.append("Sin sinopsis")



    # Se forma un diccionario con los resultados
    ficha_tecnica = {}

    for nombre, dato in zip(nombres, datos):
        ficha_tecnica[nombre] = dato


    return ficha_tecnica


# Proceso para pequeñas/medianas (lectura completa)
def scrape_editorials(editorials_names, page_intervals):
    
    # Iniciando driver
    driver = create_driver()

    try:
        for name, interval in zip(editorials_names, page_intervals):
            print(f"------------------ Editorial {name} ------------------")
            print("Leyendo páginas del catálogo...")
            libros = search_editorial(name, interval[0], interval[1])    
            print(f"✓ {len(libros)} libros encontrados para {name}")        
            
            print("Extrayendo fichas técnicas...")
            lista_libros = []
            
            for libro in libros:
                ficha = extract_data(libro["url"] + "#fichaTecnica")
                if not "Traducción" in ficha.keys():
                    ficha["Traducción"] = "Sin traducción"
                else:
                    pass                

                ficha["Título"] = libro["titulo"]
                ficha["Precio"] = libro["precio"]
                ficha['URL'] = libro['url']
                lista_libros.append(ficha)


            print(f"✓ {len(lista_libros)} fichas técnicas guardadas para {name}")
            print(f"Escribiendo catalogo_{name}.csv...")
            fieldnames = lista_libros[0].keys()
            

            with open(f"data/catalogo_{name}.csv", "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames = fieldnames)
                writer.writeheader()
                writer.writerows(lista_libros)
        
            print(f"Archivo catalogo_{name}.csv escrito y guardaddo.\n")
        
    finally:
        driver.quit()


# Proceso para grandes (lectura completa)
def scrape_big_editorials(editorials_names, year_intervals):
    
    # Iniciando driver
    driver = create_driver()

    try:
        for name, interval in zip(editorials_names, year_intervals):
            print(f"------------------ Editorial {name} ------------------")
            print("Leyendo páginas del catálogo...")
            libros = search_big_editorial(name, interval[0], interval[1])    
            print(f"✓ {len(libros)} libros encontrados para {name}")        
            
            print("Extrayendo fichas técnicas...")
            lista_libros = []
            
            for libro in libros:
                ficha = extract_data(libro["url"] + "#fichaTecnica")
                if not "Traducción" in ficha.keys():
                    ficha["Traducción"] = "Sin traducción"
                else:
                    pass                

                ficha["Título"] = libro["titulo"]
                ficha["Precio"] = libro["precio"]
                ficha['URL'] = libro['url']
                lista_libros.append(ficha)


            print(f"✓ {len(lista_libros)} fichas técnicas guardadas para {name}")
            print(f"Escribiendo catalogo_{name}.csv...")
            fieldnames = lista_libros[0].keys()
            

            with open(f"data/catalogo_{name}.csv", "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames = fieldnames)
                writer.writeheader()
                writer.writerows(lista_libros)
        
            print(f"Archivo catalogo_{name}.csv escrito y guardaddo.\n")
        
    finally:
        driver.quit()


def flow_control(editorials = "editorials.json", saved_data = "estado.json"):

    if os.path.exists(editorials):
        with open(editorials, "r", encoding="utf-8") as f:
            editorials_info = json.load(f)
    else:
        print("No se encuentra el archivo.")

    if os.path.exists(saved_data):
        with open(saved_data, "r", encoding="utf-8") as f:
            procesadas = json.load(f)
    else:
        procesadas = {}

    for editorial in editorials_info:
        if editorial in procesadas and editorials_info[editorial]["intervalo"][1] == procesadas[editorial]["intervalo"][1]:
            print(f"{editorial}: completa, saltando...")
            continue
        elif editorial in procesadas and editorials_info[editorial]["intervalo"][1] < procesadas[editorial]["intervalo"][1]:
            respuesta = input(f"{editorial}: proceso comenzado. ¿Continuar desde último guardado {(procesadas[editorial]["intervalo"][1])}? [Y/N]")
        elif not editorial in procesadas:
            respuesta = input(f"{editorial}: aún no procesada. ¿Desea comenzar el proceso? [Y/N]")

        if respuesta =="N":
            print("\n")
        elif respuesta == "Y":
            if not editorial in procesadas:
                procesadas[editorial] = {
                    "intervalo": [1, 1]
                }

            limite = input(f"Introduzca límite (menor o igual a {editorials_info[editorial]["intervalo"][1]}): ")
                
            if editorials_info["grande"]:
                scrape_big_editorials([editorials_info["id"]], [editorial], [[procesadas[editorial]["intervalo"][1], limite]])
            else:
                scrape_editorials([editorials_info["id"]], [editorial], [[procesadas[editorial]["intervalo"][1], limite]])

            procesadas[editorial]["intervalo"][1] = limite
            print("\n")

In [1]:
# ======================================================================================
# OBTENCIÓN DE DATOS (VERSIÓN FINAL Y CORREGIDA)
# ======================================================================================

import os
import json
import csv
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# ======================================================================================
# DRIVER
# ======================================================================================

def crear_driver():
    """Crea y devuelve una instancia de Chrome en modo headless."""

    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=opciones)


# Variable global del driver, se inicializa en control_flujo()
driver = None


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL NORMAL (por páginas)
# ======================================================================================

def buscar_editorial(id_editorial, nombre_editorial, pag_inicio, pag_fin):
    """
    Recorre el catálogo de una editorial normal (hasta 200 páginas) iterando de pag_inicio a pag_fin.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **pag_inicio:** página del catálogo web donde empezar a hacer scraping
    * **pag_final:** última página del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionarios con los resultados del scraping
    """
    libros = []
    num_pagina = pag_inicio

    while num_pagina <= pag_fin:
        url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?page={num_pagina}"

        # Mostrar progreso solo hasta la página 10 para no llenar la consola
        if num_pagina < 10:
            print(f"Página {num_pagina}...")
        elif num_pagina == 10:
            print("Página 10 y más...")

        driver.get(url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        elementos = soup.select("h2 a")

        # Si no hay libros en la página, se acabó el catálogo
        if not elementos:
            print("Sin más resultados.")
            break

        for h2 in soup.select("h2"):
            a = h2.find("a")
            if not a:
                continue

            titulo = a.get_text(strip=True)

            # El autor está en el h3 inmediatamente después del h2
            h3 = h2.find_next_sibling("h3")
            autor = h3.get_text(strip=True) if h3 else ""

            # El precio está en el primer <strong> después del h2
            etiqueta_precio = h2.find_next("strong")
            precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

            url_libro = a["href"] if a.get("href") else ""

            libros.append({
                "editorial": nombre_editorial,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        num_pagina += 1
        time.sleep(0.5)

    return libros


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL GRANDE (por años, para superar el límite de 200 págs)
# ======================================================================================

def buscar_editorial_grande(id_editorial, nombre_editorial, anio_inicio, anio_fin):
    """
    Recorre el catálogo de una editorial grande filtrando por año, lo que permite superar el límite de 200 páginas por búsqueda.
    Para cada año itera todas las páginas disponibles hasta que no haya resultados.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **anio_inicio:** página (del año) del catálogo web donde empezar a hacer scraping
    * **anio_final:** última página (del año) del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionario con los resultados del scraping
    """
    libros = []

    for anio in range(anio_inicio, anio_fin + 1):
        print(f"Año {anio}...")
        num_pagina = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?anios={anio}&page={num_pagina}"
            driver.get(url)
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            elementos = soup.select("h2 a")
            
            # Si no hay libros, este año ya no tiene más páginas
            if not elementos:
                print(f"Sin más resultados en {anio}, página {num_pagina}.")
                break

            for h2 in soup.select("h2"):
                a = h2.find("a")
                if not a:
                    continue

                titulo = a.get_text(strip=True)

                h3 = h2.find_next_sibling("h3")
                autor = h3.get_text(strip=True) if h3 else ""

                etiqueta_precio = h2.find_next("strong")
                precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

                url_libro = a["href"] if a.get("href") else ""

                libros.append({
                    "editorial": nombre_editorial,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            num_pagina += 1
            time.sleep(0.5)

    return libros


# ======================================================================================
# EXTRACCIÓN DE FICHA TÉCNICA Y SINOPSIS
# ======================================================================================

def extraer_datos(url):
    """
    Accede a la ficha de un libro y extrae todos los datos técnicos (ISBN, páginas, formato, etc.) y la sinopsis completa.
    Devuelve un diccionario con todos los campos encontrados.

    Parámetros:
    * **url:** link de la página de la ficha técnica

    Outputs: 
    * Diccionario con los resultados del scraping
    """
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # La ficha técnica está dentro de elementos <dl class="datos-tecnicos">
    secciones = soup.find_all("dl", class_="datos-tecnicos")

    nombres = []  # nombres de los campos (dt)
    datos = []    # valores de los campos (dd)

    for seccion in secciones:
        etiquetas_nombre = seccion.find_all("dt")
        etiquetas_dato = seccion.find_all("dd")

        # Extraer nombres de los campos técnicos
        for nombre in etiquetas_nombre:
            nombres.append(nombre.get_text(strip=True).replace(":", "").strip())

        # Extraer valores de los campos técnicos
        # Hay tres posibles estructuras dentro de cada <dd>:
        for dato in etiquetas_dato:

            # Caso 1: el valor está en uno o varios <a> (ej: categorías, editorial)
            enlaces = dato.find_all("a")
            if enlaces:
                lista_valores = [enlace.get_text(strip=True) for enlace in enlaces]
                # Si hay un solo valor lo guardamos como string, si hay varios como lista
                datos.append(lista_valores[0] if len(lista_valores) == 1 else lista_valores)
                continue

            # Caso 2: el valor está en un <span> (ej: idioma)
            span = dato.find("span")
            if span:
                datos.append(span.get_text(strip=True))
                continue

            # Caso 3: el valor está directamente en el <dd> (ej: dimensiones, páginas)
            # Usamos split/join para limpiar espacios y saltos de línea extra
            texto = dato.get_text()
            datos.append(" ".join(texto.split()))

    # Sinopsis completa: está en un div separado fuera de la ficha técnica
    sinopsis = soup.find("div", id="collapseSynopsis")
    nombres.append("Sinopsis")
    if sinopsis:
        # Extraemos párrafo a párrafo y los unimos con " | "
        parrafos = [p.get_text(strip=True) for p in sinopsis.find_all("p")]
        datos.append(" | ".join(parrafos))
    else:
        datos.append("Sin sinopsis")

    # Construir diccionario emparejando cada nombre con su dato
    ficha_tecnica = {}
    for nombre, dato in zip(nombres, datos):
        ficha_tecnica[nombre] = dato

    return ficha_tecnica


# ======================================================================================
# SCRAPING COMPLETO DE UNA EDITORIAL (búsqueda + fichas + guardado CSV)
# ======================================================================================

def scrapear_editorial(id_editorial, nombre_editorial, inicio, fin, es_grande):
    """
    Orquesta el proceso completo para una editorial:
    1. Busca todos los libros del catálogo en el intervalo indicado.
    2. Entra en cada ficha técnica y extrae los datos.
    3. Guarda el resultado en un CSV en la carpeta 'data/'.

    Parámetros:
    * **id_editorial:** identificador de la URL (ej: "debolsillo_179709")
    * **nombre_editorial:** nombre legible (ej: "DEBOLSILLO")
    * **inicio:** página o año de inicio
    * **fin:** página o año de fin
    * **es_grande:** True si es editorial grande (búsqueda por años)

    Output:
    * Archivo .csv con las fichas técnicas de los libros
    """
    print(f"\n{'='*60}")
    print(f"Editorial: {nombre_editorial}")
    print(f"{'='*60}")

    # Fase 1: recoger listado de libros
    print("Leyendo catálogo...")
    if es_grande:
        libros = buscar_editorial_grande(id_editorial, nombre_editorial, inicio, fin)
    else:
        libros = buscar_editorial(id_editorial, nombre_editorial, inicio, fin)

    print(f"✓ {len(libros)} libros encontrados.")

    if not libros:
        print("No hay libros que procesar.")
        return

    # Fase 2: extraer fichas técnicas
    print("Extrayendo fichas técnicas...")
    lista_libros = []

    for i, libro in enumerate(libros):
        print(f"  [{i+1}/{len(libros)}] {libro['titulo'][:50]}...")

        ficha = extraer_datos(libro["url"] + "#fichaTecnica")

        # Añadir campo traducción si no existe
        if "Traducción" not in ficha:
            ficha["Traducción"] = "Sin traducción"

        # Añadir datos del catálogo a la ficha
        ficha["Título"] = libro["titulo"]
        ficha["Precio"] = libro["precio"]
        ficha["URL"] = libro["url"]

        lista_libros.append(ficha)

    # Fase 3: guardar JSON
    os.makedirs("data", exist_ok=True)
    ruta_json = f"data/catalogos/catalogo_{nombre_editorial.lower()}.json"
    campos = lista_libros[0].keys()

    # Revisión del catálogo
    if os.path.exists(ruta_json):
        with open(ruta_json, "r", encoding="utf-8") as f:
            libros_existentes = json.load(f)
    else:
        libros_existentes = []

    # Añadir nuevos libros al final
    libros_existentes.extend(lista_libros)

    # Guardar el resultado completo
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(libros_existentes, f, ensure_ascii=False, indent=2)

    print(f"✓ {len(lista_libros)} fichas guardadas en {ruta_json}.")


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo(ruta_editoriales="data/json/editoriales.json", ruta_estado="data/json/estado.json"):
    """
    Función principal que gestiona el flujo interactivo del scraping.

    Lee el fichero de editoriales (editoriales.json) con la información de cada una,
    consulta el estado guardado (estado.json) para saber cuáles ya están procesadas,
    y pregunta al usuario qué hacer con cada una.

    Estructura esperada de editoriales.json:
    {
        "debolsillo": {
            "id": "debolsillo_179709",
            "grande": false,
            "intervalo_max": [1, 200]   <- páginas si normal, años si grande
        },
        "espasa": {
            "id": "espasa_76490",
            "grande": true,
            "intervalo_max": [1990, 2024]
        }
    }

    Estructura de estado.json (se genera automáticamente):
    {
        "debolsillo": {
            "ultimo": 200    <- última página/año procesada
        }
    }
    """
    global driver

    # Cargar información de editoriales
    if not os.path.exists(ruta_editoriales):
        print(f"Error: no se encuentra '{ruta_editoriales}'.")
        return

    with open(ruta_editoriales, "r", encoding="utf-8") as f:
        info_editoriales = json.load(f)

    # Cargar estado previo si existe, o empezar desde cero
    if os.path.exists(ruta_estado):
        with open(ruta_estado, "r", encoding="utf-8") as f:
            estado = json.load(f)
    else:
        estado = {}

    # Iniciar el driver una sola vez para toda la sesión
    print("Iniciando navegador...")
    driver = crear_driver()
    print("✓ Navegador listo.\n")

    try:
        for nombre, info in info_editoriales.items():
            maximo = info["intervalo"][1]
            es_grande = info["grande"]
            id_editorial = info["id"]

            # Comprobar si ya está completamente procesada
            if nombre in estado and estado[nombre]["ultimo"] >= maximo:
                print(f"{nombre}: ya procesada completamente, saltando...")
                continue

            # Determinar punto de inicio (desde el principio o desde donde se dejó)
            if nombre in estado:
                ultimo_guardado = estado[nombre]["ultimo"]
                inicio_sugerido = ultimo_guardado+1 # retomamos desde el último guardado
                msg = f"{nombre}: proceso iniciado. Último guardado en {ultimo_guardado}. ¿Continuar? [Y/N]: "
            else:
                inicio_sugerido = info["intervalo"][0]
                msg = f"{nombre}: aún no procesada. ¿Comenzar? [Y/N]: "

            respuesta = input(msg).strip().upper()

            if respuesta == "N":
                print(f"Saltando {nombre}.\n")
                continue

            elif respuesta == "Y":
                # Pedir intervalo al usuario
                tipo = "año" if es_grande else "página"
                print(f"Inicio sugerido: {inicio_sugerido} | Máximo disponible: {maximo}")

                entrada_inicio = input(f"Introduce {tipo} de inicio [{inicio_sugerido}]: ").strip()
                entrada_fin = input(f"Introduce {tipo} de fin (máx. {maximo}): ").strip()

                # Usar valores sugeridos si el usuario no introduce nada
                inicio = int(entrada_inicio) if entrada_inicio else inicio_sugerido
                fin = min(int(entrada_fin), maximo)  # nunca superar el máximo

                # Ejecutar el scraping
                scrapear_editorial(id_editorial, nombre, inicio, fin, es_grande)

                # Actualizar y guardar estado
                if nombre not in estado:
                    estado[nombre] = {}
                estado[nombre]["ultimo"] = fin

                with open(ruta_estado, "w", encoding="utf-8") as f:
                    json.dump(estado, f, ensure_ascii=False, indent=2)

                print(f"Estado guardado: {nombre} → hasta {tipo} {fin}.\n")

            else:
                print("Respuesta no válida, saltando.\n")

    finally:
        # Cerrar el navegador siempre, aunque haya errores
        driver.quit()
        print("\nNavegador cerrado.")


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

if __name__ == "__main__":
    control_flujo()

Iniciando navegador...
✓ Navegador listo.

Planeta: ya procesada completamente, saltando...
Espasa: ya procesada completamente, saltando...
Seix Barral: ya procesada completamente, saltando...
Saltando Destino.

Booket: ya procesada completamente, saltando...
Tusquets: ya procesada completamente, saltando...
Alfaguara: ya procesada completamente, saltando...
Saltando Plaza & Janes.

Debolsillo: ya procesada completamente, saltando...
Saltando Lumen.

Saltando Reservoir Books.

Saltando Grijalbo.

Saltando HarperCollins.

Saltando Harlequin Ibérica.

Saltando Harlequin.

Saltando Devir Iberia.

Saltando Ediciones Iberia.

Anagrama: ya procesada completamente, saltando...
Editorial Anagrama: ya procesada completamente, saltando...
Acantilado: ya procesada completamente, saltando...
Saltando Impedimenta.

Inicio sugerido: 1 | Máximo disponible: 57

Editorial: Siruela
Leyendo catálogo...
Página 1...
Página 2...
Página 3...
Página 4...
Página 5...
Página 6...
Página 7...
Página 8...
Página 

ValueError: invalid literal for int() with base 10: ''

¿Por qué se guarda en JSON? Porque no todos los libros tienen la misma información en la ficha técnica. Si se hace un .csv al que le falte alguna columna y luego se quiera incluir algún libro que síque tenga esa columna, daría error. Esto es común con los datos físicos de los libros antiguos.

Lista de librerías / sellos con id, total libros, intervalo anios/páginas:
{
    "errata+natural":
    "traficante+de+sueños":
    "libros+del+zorro+rojo":
    "nórdica+books":
    "fulgencio+pimentel":
    "las+afueras":
    "la+caja+books":
    "ediciones+menguante":
    "dos+bigotes":
    "plastic+books":
}




In [ ]:
# ======================================================================================
# CREACIÓN DEL DATAFRAME BASE
# ======================================================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path


def crear_df(ruta_catalogos="data/catalogos", ruta_guardado="data/parquet/lista_libros.parquet"):
    path = Path(ruta_catalogos)
    df = pd.DataFrame({})
    jsons = []

    print("="*50, "\nCreando DataFrame con todos los libros\n", "="*50)
    for archivo in path.iterdir():
        if archivo.is_file():
            print(f"Añadiendo {archivo.name}")
            editorial = pd.read_json(archivo.absolute())
            jsons.append(editorial)

    df = pd.concat(jsons, axis=0)

    # Borrar filas repetidas
    print("Catálogos convertidos a DataFrame. Eliminando filas duplicadas...")
    df.drop_duplicates(subset=df.columns, keep='first', inplace=True)

    # Creando columnas que van a ser rellenadas con la API de OpenLibrary
    df.columns = df.columns.str.strip().str.lower().str.translate(str.maketrans({"á": "a", "é": "e", "í": "i", "ó":"o", "ú": "u", "º": "", " ": "_"}))
    

    # Creando .parquet
    df.to_parquet(ruta_guardado, index=False)

    print("DataFrame guardado con éxito.")

def limpiar_df(df: pd.DataFrame, cats_borrar: set):
    df_clean = df.copy() 

    # Dejar libros solo en español
    df_clean = df_clean[df_clean['idioma_de_publicacion'] == 'Castellano']

    # Eliminar duplicados (tienen el mismo ean)
    df_clean['titulo'] = df_clean['titulo'].str.strip().str.capitalize()
    df_clean.drop_duplicates(subset=['ean'], inplace=True)

    # Eliminar libros con datos importantes faltantes
    df_clean.drop(subset=['titulo', 'autor', 'ean'], inplace=True)

    # Eliminar categorías no interesantes
    # 1. Convetir categorías a lista
    df_clean['categorias'] = df_clean['categorias'].apply(lambda x: x.to_list() if x.dtype == str else x)
    mask = df_clean['categorias'].apply(lambda x: any(c in cats_borrar for c in x) if isinstance(x, list) else False)
    df_clean = df_clean[~mask]

    # Eliminar libros únicos (tener cuidado con recopilaciones de teatros, antologías...)


    # Regularizar dimensiones (ver a cuantos libros les faltan estas medidas y luego actuar en consecuencia)

formato de búsqueda en API de OpenLibrary: http://openlibrary.org/api/volumes/brief/isbn/9788437622149

A tener en cuenta: 
* details (dict):
    * contributors (list of dicts)
        * role(key), name(key) (ej.: "contributors": [
            {
              "role": "Editor",
              "name": "John Jay Allen"
            }
          ])
    * subjects (dict)

Para ratings: https://openlibrary.org/search.json?isbn=9788437622149&fields=rating*

Ejemplo de uso de API dado por OpenLibrary:
import requests
url = "https://openlibrary.org/search.json?q=test"
headers = {
    "User-Agent": "MyAppName/1.0 (myemail@example.com)"
}
response = requests.get(url, headers=headers)

In [ ]:
# ======================================================================================
# ACCESO A API DE OPENLIBRARY
# ======================================================================================

import requests
import json
import pandas as pd
from pathlib import Path
import re


# ======================================================================================
# LECTURA DE DATOS
# ======================================================================================

# --- Contribuidores (editor, traductor...)
def leer_brief(isbn):   

    # Llamada para contribuidores
    url = f"http://openlibrary.org/api/volumes/brief/isbn/{isbn}"

    response = requests.get(url)

    if response.status_code == 200:
        api1 = response.json()

        if api1 == []:
            # No encuentra el libro
            return True
        
        contenido = api1["records"]

        # clave tipo /books/OL9130631M
        clave = list(contenido.keys())[0]
        
        # Edición exacta
        edicion = contenido[clave]["details"]["details"]["edition_name"]

        # Editores
        contribuidores = contenido[clave]["details"]["details"]["contributors"]
        editores = []
        if contribuidores != "":
            for contr in contribuidores:
                if contr['role'] == 'Editor':
                    editores.append(contr["name"])

        # Peso
        peso = contenido[clave]["details"]["details"]['weight']

        # Dimensiones
        dim = contenido[clave]["details"]["details"]['physical_dimensions']

        # Formato
        formato = contenido[clave]["details"]["details"]['physical_format']

        # Categorías
        total_cats = contenido[clave]["data"]["subjects"]
        categorias = []
        for cat_dict in total_cats:
            categorias.append(cat_dict["name"])
        categorias += contenido[clave]["details"]["details"]['subjects']
        categorias.apply(lambda x: x.translate(str.maketrans({"-":"", "/": "", "&": "and"})).strip())
        

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return {'edicion': edicion, 'editores': editores, 'peso': peso, 'dim': dim, 'formato': formato, 'subcategorias': categorias}

# --- Ratings
def leer_ratings(isbn):

    # Llamada para ratings
    url = f"https://openlibrary.org/search.json?isbn={isbn}&fields=rating*"

    response = requests.get(url)

    if response.status_code == 200:
        api2 = response.json()
        ratings = api2["docs"][0]

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return ratings

# --- Sinopsis (ver como conseguirla de otro sitio)


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo_api(ruta_catalogos="data/catalogos"): # cambiarlo para que te permita elegir cuántos libros hay que 

    print("Iniciando búsqueda en API...")
    catalogos = [f for f in Path(ruta_catalogos).iterdir() if f.is_file()]

    for ruta_cat in catalogos:
        print(f"\nLeyendo {ruta_cat.name}...")

        with open(ruta_cat, "r", encoding="utf-8") as f:
            catalogo = json.load(f)

        if 'openl' in catalogo[0].keys():
            print("Catálogo completo.")
            continue 
        
        key = input("Continuar [Y/N]?")    

        if key=='N':
            continue

        print(f"Comenzando con {ruta_cat.name}. Total de libros a buscar: {len(catalogo)+1}...")
        for libro in range(len(catalogo)):
            print(f"[{libro+1}/{len(catalogo)+1}]")
            ean = catalogo[libro]['ean']

            brief = leer_brief(ean)
            ratings = leer_ratings(ean)

            catalogo[libro].update({
                **brief, 
                **ratings,
                'openl': True
            })
        print("Catálogo terminado.")

        with open(ruta_cat, "w", encoding="utf-8") as f:
                    json.dump(catalogo, f, ensure_ascii=False, indent=2)


In [ ]:
x, y = leer_brief(9788437622149)
print(x)
print(y)

[{'role': 'Editor', 'name': 'John Jay Allen'}]

In [ ]:
leer_ratings(9788437622149)

{'ratings_average': 3.8947368,
 'ratings_sortable': 3.5598314,
 'ratings_count_1': 4,
 'ratings_count_2': 3,
 'ratings_count_3': 13,
 'ratings_count_4': 12,
 'ratings_count_5': 25,
 'ratings_count': 57}

La idea es que, una vez construidos los diccionarios, se pasen a una función que busque cada libro por su EAN (ISBN sin guiones) y cree una nueva key con cada dato relevante: editor (obviar traductor si ya lo tiene), rating_average y rating_count. Esto sería el paso previo a construir el dataset (la informaicón de editoriales se hará haciendo merge con otra tabla específica de eso, al igual que si se quiere añadir información de traductores, editores...)

In [ ]:
# ======================================================================================
# DATOS DE SPI
# ======================================================================================

from functools import reduce

def merge_spi(lista_editoriales, ruta_ed="data/json/editoriales.json", ruta_spi="data/bronze/spi"):
    
    with open(ruta_ed, "r", encoding="utf-8") as f:
        editoriales = json.load(f)

    clasificaciones = list(Path(ruta_spi).iterdir())
    dfs = []
    for ruta_cla in clasificaciones:
        print(f"\nIncluyendo {ruta_cla.name}...")
        with open(ruta_cla, "r", encoding="utf-8") as f:
            dfs.append(pd.DataFrame(json.load(f)))

    df = reduce(lambda left, right: left.merge(right, on="Editorial", how='outer'), dfs)

    nombre_spi = {editoriales[ed]["nombre_spi"]: ed for ed in lista_editoriales}
    id_dict = {editoriales[ed]["nombre_spi"]: editoriales[ed]["id"] for ed in lista_editoriales}

    df_selection = df[df['Editorial'].isin(nombre_spi.keys())].copy()
    df_selection['Editorial'] = df_selection["Editorial"].map(nombre_spi)
    df_selection['id'] = df_selection["Editorial"].map(id_dict)

    return df_selection